In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CAT = 'northbridge'
B_TX, B_LG = f'{CAT}.bronze.transactions', f'{CAT}.bronze.ledger'
S_TX, S_RJ, S_LG = f'{CAT}.silver.transactions', f'{CAT}.silver.transactions_rejected', f'{CAT}.silver.ledger'
DQ = f'{CAT}.ops.dq_results'
VALID_TYPES = ['PAYMENT', 'TRANSFER', 'CASH_OUT', 'DEBIT', 'CASH_IN']

In [0]:
spark.sql(f"""CREATE TABLE IF NOT EXISTS {S_TX} (
  txn_id STRING, day INT, step INT, txn_type STRING, amount DECIMAL(18,2),
  orig_id STRING, old_bal_orig DECIMAL(18,2), new_bal_orig DECIMAL(18,2),
  dest_id STRING, old_bal_dest DECIMAL(18,2), new_bal_dest DECIMAL(18,2),
  is_fraud_label INT)""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {S_RJ} (
  txn_id STRING, day INT, step STRING, type STRING, amount STRING,
  nameOrig STRING, oldbalanceOrg STRING, newbalanceOrig STRING,
  nameDest STRING, oldbalanceDest STRING, newbalanceDest STRING,
  isFraud STRING, isFlaggedFraud STRING,
  reject_reason STRING, _ingest_ts TIMESTAMP, _source_file STRING)""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {S_LG} (
  txn_id STRING, account_id STRING, txn_type STRING,
  ledger_amount DECIMAL(18,2), day INT)""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {DQ} (
  check_name STRING, run_ts TIMESTAMP, status STRING, row_count BIGINT, notes STRING)""")

In [0]:
def silver_transactions(days):
    day_list = ','.join(str(d) for d in days)

    b = (spark.table(B_TX)
         .withColumn('day', F.col('day').cast('int'))
         .filter(F.col('day').isin(days)))

    t = (b
      .withColumn('step_i',  F.expr('try_cast(step as int)'))
      .withColumn('amount_d', F.expr('try_cast(amount as decimal(18,2))'))
      .withColumn('obo_d',   F.expr('try_cast(oldbalanceOrg as decimal(18,2))'))
      .withColumn('nbo_d',   F.expr('try_cast(newbalanceOrig as decimal(18,2))'))
      .withColumn('obd_d',   F.expr('try_cast(oldbalanceDest as decimal(18,2))'))
      .withColumn('nbd_d',   F.expr('try_cast(newbalanceDest as decimal(18,2))'))
      .withColumn('fraud_i', F.expr('try_cast(isFraud as int)')))

    reason = (F.when(F.col('txn_id').isNull(), 'NULL_TXN_ID')
      .when(F.col('amount_d').isNull() | (F.col('amount_d') <= 0), 'INVALID_AMOUNT')
      .when(F.col('type').isNull() | ~F.col('type').isin(VALID_TYPES), 'INVALID_TYPE')
      .when(F.col('step_i').isNull() | (F.col('step_i') < 1) | (F.col('step_i') > 744), 'INVALID_STEP')
      .when(F.col('obo_d').isNull() | F.col('nbo_d').isNull()
            | F.col('obd_d').isNull() | F.col('nbd_d').isNull(), 'NULL_BALANCE'))

    t = t.withColumn('reject_reason', reason)
    bad  = t.filter(F.col('reject_reason').isNotNull())
    good = t.filter(F.col('reject_reason').isNull())

    w = Window.partitionBy('txn_id').orderBy('_ingest_ts', '_source_file')
    good = good.withColumn('rn', F.row_number().over(w))
    dups  = good.filter('rn > 1').withColumn('reject_reason', F.lit('DUPLICATE_TXN_ID'))
    clean = good.filter('rn = 1')

    # 1) rejected rows: delete-then-append per day so reruns do not double up
    raw_cols = ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
                'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']
    rejected = (bad.unionByName(dups.drop('rn'))
        .select(F.col('txn_id').cast('string'), 'day',
                *[F.col(c).cast('string') for c in raw_cols],
                'reject_reason', '_ingest_ts', '_source_file'))
    spark.sql(f"DELETE FROM {S_RJ} WHERE day IN ({day_list})")
    rejected.write.mode('append').saveAsTable(S_RJ)

    # 2) clean rows: MERGE, so loading the same day twice changes nothing
    (clean.select('txn_id', 'day',
          F.col('step_i').alias('step'), F.col('type').alias('txn_type'),
          F.col('amount_d').alias('amount'),
          F.col('nameOrig').alias('orig_id'),
          F.col('obo_d').alias('old_bal_orig'), F.col('nbo_d').alias('new_bal_orig'),
          F.col('nameDest').alias('dest_id'),
          F.col('obd_d').alias('old_bal_dest'), F.col('nbd_d').alias('new_bal_dest'),
          F.col('fraud_i').alias('is_fraud_label'))
      .createOrReplaceTempView('staged_clean'))

    spark.sql(f"""MERGE INTO {S_TX} t USING staged_clean s
                  ON t.txn_id = s.txn_id
                  WHEN NOT MATCHED THEN INSERT *""")
    print('transactions done for days', days)

In [0]:
def silver_ledger(days):
    b = (spark.table(B_LG)
         .withColumn('day', F.col('day').cast('int'))
         .filter(F.col('day').isin(days)))

    w = Window.partitionBy('txn_id').orderBy('_ingest_ts')
    (b.withColumn('ledger_amount', F.expr('try_cast(ledger_amount as decimal(18,2))'))
      .filter(F.col('txn_id').isNotNull() & F.col('ledger_amount').isNotNull())
      .withColumn('rn', F.row_number().over(w)).filter('rn = 1')
      .select('txn_id', 'account_id', 'txn_type', 'ledger_amount', 'day')
      .createOrReplaceTempView('staged_ledger'))

    spark.sql(f"""MERGE INTO {S_LG} t USING staged_ledger s
                  ON t.txn_id = s.txn_id
                  WHEN NOT MATCHED THEN INSERT *""")
    print('ledger done for days', days)

In [0]:
for start in range(8, 32, 6):
    days = list(range(start, min(start + 6, 32)))
    silver_transactions(days)
    silver_ledger(days)

In [0]:
DAYS = list(range(1, 8))
silver_transactions(DAYS)
silver_ledger(DAYS)

In [0]:
def cnt(table, name):
    return (spark.table(table).withColumn('day', F.col('day').cast('int'))
            .groupBy('day').count().withColumnRenamed('count', name))

chk = (cnt(B_TX, 'bronze').join(cnt(S_TX, 'silver'), 'day', 'left')
       .join(cnt(S_RJ, 'rejected'), 'day', 'left').fillna(0)
       .withColumn('diff', F.col('bronze') - F.col('silver') - F.col('rejected'))
       .orderBy('day'))
display(chk)

In [0]:
display(spark.sql(f"SELECT reject_reason, count(*) AS rows FROM {S_RJ} GROUP BY reject_reason ORDER BY rows DESC"))

display(spark.sql(f"""SELECT count(*) AS total, count(DISTINCT txn_id) AS distinct_ids,
                      sum(CASE WHEN amount <= 0 THEN 1 ELSE 0 END) AS bad_amounts
                      FROM {S_TX}"""))

In [0]:
def log_dq(name, ok, n, notes=''):
    spark.createDataFrame([(name, 'PASS' if ok else 'FAIL', n, notes)],
                          'check_name string, status string, row_count long, notes string') \
         .withColumn('run_ts', F.current_timestamp()) \
         .select('check_name', 'run_ts', 'status', 'row_count', 'notes') \
         .write.mode('append').saveAsTable(DQ)

balanced = chk.filter('diff != 0').count() == 0
log_dq('bronze_eq_silver_plus_rejected', balanced, chk.agg(F.sum('bronze')).first()[0])

s = spark.table(S_TX)
log_dq('silver_tx_no_duplicate_ids', s.count() == s.select('txn_id').distinct().count(), s.count())

display(spark.table(DQ))

In [0]:
before = (spark.table(S_TX).count(), spark.table(S_RJ).count(), spark.table(S_LG).count())
silver_transactions(DAYS); silver_ledger(DAYS)
after  = (spark.table(S_TX).count(), spark.table(S_RJ).count(), spark.table(S_LG).count())
print(before, after, before == after)

In [0]:
display(spark.sql(f"""SELECT (SELECT count(*) FROM {B_LG}) AS bronze_ledger,
                             (SELECT count(*) FROM {S_LG}) AS silver_ledger"""))

In [0]:
spark.sql(f"""CREATE OR REPLACE FUNCTION {CAT}.silver.mask_phone(phone STRING)
RETURN CASE WHEN is_account_group_member('pii_readers') THEN phone
            ELSE concat('XXXXXX', right(phone, 4)) END""")

display(spark.sql(f"SELECT {CAT}.silver.mask_phone('9123456789') AS masked"))